Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
CC3084 - Data Science - sección 40

Integrantes:  
- Cristian Túnchez (231359)
- Dulce Ambrosio (231143)
- Daniel Chet (231177)
- Javier Linares (231135)

# 01. Carga y limpieza de los datos

**MAP - Charting Student Math Misunderstandings**

En este cuaderno se realiza el primer contacto con los datos de la competencia. El objetivo es responder tres preguntas antes de cualquier análisis: qué contiene el conjunto de datos, qué tan confiable es, y qué transformaciones necesita para poder ser analizado.

El trabajo se organiza de la siguiente manera:

1. **Carga y descripción de la estructura.** Cuántas observaciones y variables hay, y de qué tipo es cada una.
2. **Diagnóstico de calidad.** Valores faltantes, duplicados y consistencia interna.
3. **Limpieza y construcción de variables.** Las transformaciones necesarias y las variables nuevas que se derivan del texto.

El resultado se guarda en `data/processed/` y alimenta el cuaderno `02_eda.ipynb`.

## 1. Preparación del entorno

Se importan las bibliotecas necesarias y se añade la carpeta `src/` a la ruta de búsqueda de
Python para poder reutilizar las funciones del proyecto.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
sys.path.append(str(RAIZ / "src"))

from utils import DATA_RAW, guardar, set_estilo

set_estilo()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

print("Carpeta de datos crudos:", DATA_RAW)

Carpeta de datos crudos: /home/sebas/github/Data-Science/math-misconception-map/data/raw


## 2. Carga de los archivos

La competencia entrega tres archivos. El de entrenamiento es el único que trae las etiquetas y por tanto es el que se analiza. Los otros dos se cargan solo para entender el formato que se espera en la entrega.

In [2]:
train = pd.read_csv(DATA_RAW / "train.csv")
test = pd.read_csv(DATA_RAW / "test.csv")
sample_submission = pd.read_csv(DATA_RAW / "sample_submission.csv")

for nombre, df in [("train", train), ("test", test), ("sample_submission", sample_submission)]:
    print(f"{nombre:20s} filas: {df.shape[0]:>6,}   columnas: {df.shape[1]}")

train                filas: 36,696   columnas: 7
test                 filas:      3   columnas: 5
sample_submission    filas:      3   columnas: 2


In [3]:
print("Columnas de train:", list(train.columns))
print("Columnas de test: ", list(test.columns))
print()
print("Formato de entrega esperado:")
sample_submission

Columnas de train: ['row_id', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation', 'Category', 'Misconception']
Columnas de test:  ['row_id', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation']

Formato de entrega esperado:


,row_id,Category:Misconception
0,36696,True_Correct:NA False_Neither:NA False_Misconception:Incomplete
1,36697,True_Correct:NA False_Neither:NA False_Misconception:Incomplete
2,36698,True_Correct:NA False_Neither:NA False_Misconception:Incomplete


### ¿Qué se observa?

El conjunto de entrenamiento tiene **36,696 observaciones y 7 variables**. El archivo de prueba solo tiene 3 filas, porque es un marcador de posición (la competencia sustituye ese archivo por uno de aproximadamente 16,000 filas al momento de evaluar). Por eso todo el análisis se hace sobre `train`.

El archivo de prueba tiene 5 columnas en lugar de 7. Le faltan justamente `Category` y `Misconception`, que son las dos variables que hay que predecir.

El formato de entrega confirma que se espera una cadena `Category:Misconception` y se admiten **hasta tres predicciones separadas por espacio**. Es decir, la evaluación reconoce que la tarea es ambigua y premia tener la respuesta correcta entre las tres primeras candidatas.

Conviene notar que la documentación de la competencia no menciona la columna `row_id`, que sí está presente. Es simplemente el identificador de la fila.

## 3. Estructura de las variables

Se revisa el tipo de dato de cada variable, cuántos valores distintos toma y cuántos valores
faltantes tiene.

In [4]:
resumen = pd.DataFrame({
    "tipo_pandas": train.dtypes.astype(str),
    "valores_unicos": train.nunique(),
    "faltantes": train.isna().sum(),
    "pct_faltante": (train.isna().mean() * 100).round(2),
})
resumen

,tipo_pandas,valores_unicos,faltantes,pct_faltante
row_id,int64,36696,0,0.00
QuestionId,int64,15,0,0.00
QuestionText,object,15,0,0.00
MC_Answer,object,49,0,0.00
StudentExplanation,object,35726,0,0.00
Category,object,6,0,0.00
Misconception,object,35,26836,73.13


In [5]:
train.head(5)

,row_id,QuestionId,QuestionText,MC_Answer,StudentExplanation,Category,Misconception
0,0,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct,NaN
1,1,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third is not shaded.,True_Correct,NaN
2,2,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to understand.",True_Neither,NaN
3,3,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 goes into everything and 3 goes into nine,True_Neither,NaN
4,4,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 out of every 3 isn't coloured,True_Correct,NaN


### Tipo y nivel de medición de cada variable

El tipo que asigna pandas no basta para saber cómo tratar una variable. Por ejemplo `QuestionId` es un número entero, pero sumarlo o promediarlo no tiene sentido porque es una etiqueta. Por eso se clasifica cada variable por su nivel de medición.

| Variable | Tipo en pandas | Nivel de medición | Descripción |
| --- | --- | --- | --- |
| `row_id` | entero | Identificador | Número de fila, único en todo el archivo |
| `QuestionId` | entero | Identificador | Identifica la pregunta diagnóstica |
| `QuestionText` | texto | Texto libre | Enunciado de la pregunta, transcrito desde una imagen |
| `MC_Answer` | texto | Nominal | Opción de respuesta que el estudiante seleccionó |
| `StudentExplanation` | texto | Texto libre | Justificación escrita por el estudiante |
| `Category` | texto | Nominal | Relación entre la respuesta elegida y la explicación |
| `Misconception` | texto | Nominal | Error conceptual específico, solo cuando aplica |

Solo hay dos variables de texto libre y ninguna variable numérica de razón.

La variable `Misconception` tiene **73.13% de valores faltantes**. Antes de decidir qué hacer con ellos hay que averiguar si son un problema de calidad o una característica del diseño.

## 4. Diagnóstico de calidad

### 4.1 Los valores faltantes de `Misconception`

La documentación indica que `Misconception` solo aplica cuando la categoría contiene un error conceptual. Se verifica si eso se cumple de forma exacta.

In [6]:
verificacion = pd.crosstab(
    train["Category"],
    train["Misconception"].notna().map({True: "tiene etiqueta", False: "vacío"}),
)
verificacion

Misconception,tiene etiqueta,vacío
Category,,
False_Correct,0,227
False_Misconception,9457,0
False_Neither,0,6542
True_Correct,0,14802
True_Misconception,403,0
True_Neither,0,5265


La correspondencia es perfecta. Las dos categorías que contienen la palabra `Misconception` tienen siempre una etiqueta, y las otras cuatro nunca la tienen.

**Decisión:** estos valores faltantes **no se imputan ni se eliminan**. No son datos perdidos, son un valor con significado propio: la ausencia de error conceptual. Se sustituyen por la categoría explícita `"NA"`, que es además la forma en que la competencia espera recibirlos en el archivo de entrega.

### 4.2 Registros duplicados

Se revisan tres niveles de duplicación, porque cada uno significa algo distinto.

In [7]:
print("Filas idénticas en todas las columnas:", train.duplicated().sum())
print("Explicaciones repetidas (texto idéntico):",
      train.duplicated(subset=["StudentExplanation"]).sum())

clave = ["QuestionId", "MC_Answer", "StudentExplanation"]
print("Combinaciones pregunta + respuesta + explicación repetidas:",
      train.duplicated(subset=clave).sum())

Filas idénticas en todas las columnas: 0
Explicaciones repetidas (texto idéntico): 970


Combinaciones pregunta + respuesta + explicación repetidas: 797


In [8]:
train["objetivo"] = train["Category"] + ":" + train["Misconception"].fillna("NA")

etiquetas_por_grupo = train.groupby(clave)["objetivo"].nunique()
print("Grupos idénticos en total:", len(etiquetas_por_grupo))
print("Grupos idénticos que recibieron más de una etiqueta:", (etiquetas_por_grupo > 1).sum())

ejemplo = etiquetas_por_grupo[etiquetas_por_grupo > 1].index[0]
train[
    (train["QuestionId"] == ejemplo[0])
    & (train["MC_Answer"] == ejemplo[1])
    & (train["StudentExplanation"] == ejemplo[2])
][["StudentExplanation", "Category", "Misconception"]]

Grupos idénticos en total: 35899
Grupos idénticos que recibieron más de una etiqueta: 61


,StudentExplanation,Category,Misconception
38,"1/3 of the 9 triangles are shaded, so 3/9=1/3.",True_Correct,NaN
39,"1/3 of the 9 triangles are shaded, so 3/9=1/3.",True_Neither,NaN


### ¿Qué se observa?

No hay ninguna fila idéntica en todas sus columnas, de modo que no hay duplicación técnica que corregir. Sí hay **970 explicaciones con texto repetido**, lo que es esperable, pues frases como "porque sí" o "lo calculé" las escriben muchos estudiantes distintos.

El hallazgo relevante está en el tercer nivel. Existen **61 grupos** en los que la misma pregunta, la misma opción elegida y exactamente la misma explicación recibieron etiquetas diferentes. Esto es ruido de anotación y una cota práctica: ningún modelo puede acertar en los dos casos de un grupo así, porque la entrada es idéntica.

**Decisión:** no se eliminan. Son observaciones legítimas y su cantidad, 61 sobre 36,696, es demasiado pequeña para afectar el análisis. Se conservan como evidencia del ruido de la tarea.

### 4.3 Consistencia entre el identificador y el texto de la pregunta

Se verifica que cada `QuestionId` corresponda siempre al mismo enunciado.

In [9]:
textos_por_pregunta = train.groupby("QuestionId")["QuestionText"].nunique()
print("Preguntas distintas:", train["QuestionId"].nunique())
print("Enunciados distintos por pregunta:", textos_por_pregunta.unique())

opciones_por_pregunta = train.groupby("QuestionId")["MC_Answer"].nunique()
print("Opciones de respuesta por pregunta:", opciones_por_pregunta.unique())
print("Opciones de respuesta distintas en total:", train["MC_Answer"].nunique())

Preguntas distintas: 15
Enunciados distintos por pregunta: [1]
Opciones de respuesta por pregunta: [4]
Opciones de respuesta distintas en total: 49


### ¿Qué se observa?

La correspondencia entre identificador y enunciado es uno a uno, así que no hay inconsistencias. Cada pregunta tiene exactamente cuatro opciones de respuesta, tal como describe el formato de pregunta diagnóstica.

**Solo hay 15 preguntas distintas** para casi 37,000 respuestas. El conjunto no es una muestra amplia de
problemas de matemática, es una muestra muy profunda de quince problemas. Esto condiciona todo lo que viene después y se analiza en detalle en el cuaderno de exploración.

También se observa que las opciones de respuesta distintas son 49 y no 60, que sería 15 preguntas por 4 opciones. Algunas opciones, típicamente números sencillos, se repiten entre preguntas distintas.

### 4.4 Consistencia de los nombres de las etiquetas

Las 35 etiquetas de error conceptual son cadenas de texto escritas por los anotadores. Se revisa si alguna aparece escrita de más de una forma.

In [10]:
etiquetas = train["Misconception"].dropna()
por_minusculas = etiquetas.str.lower().value_counts()
colisiones = {}
for nombre in etiquetas.unique():
    colisiones.setdefault(nombre.lower(), []).append(nombre)

print("Etiquetas distintas:", etiquetas.nunique())
print("Etiquetas distintas ignorando mayúsculas:", por_minusculas.size)
print()
for clave, variantes in colisiones.items():
    if len(variantes) > 1:
        print("Misma etiqueta escrita de varias formas:", variantes)

Etiquetas distintas: 35
Etiquetas distintas ignorando mayúsculas: 34

Misma etiqueta escrita de varias formas: ['Wrong_Fraction', 'Wrong_fraction']


In [11]:
duplicada = train[train["Misconception"].str.lower() == "wrong_fraction"]

print("En qué pregunta aparece cada variante:")
print(pd.crosstab(duplicada["Misconception"], duplicada["QuestionId"]).to_string())
print()
print("Ejemplos de cada variante:")
for variante in ["Wrong_fraction", "Wrong_Fraction"]:
    print(f"\n  {variante}:")
    for texto in duplicada[duplicada["Misconception"] == variante]["StudentExplanation"].head(2):
        print("   ", texto[:80])

En qué pregunta aparece cada variante:


QuestionId      31777  33471
Misconception               
Wrong_Fraction    273      0
Wrong_fraction      0    418

Ejemplos de cada variante:

  Wrong_fraction:
    24 divided by 8 equals 3 and 3 times 3 equals 9
    3/8 are worth 9. i took it away from 24

  Wrong_Fraction:
    I did 120 divided by 5 then that answer times 2
    1 / 5 equals 24 so 2 of those will be 48.


### ¿Qué se observa?

Aparece una inconsistencia que no es evidente a simple vista. La etiqueta **`Wrong_fraction`** y la etiqueta **`Wrong_Fraction`** difieren únicamente en una mayúscula, y describen el mismo error: usar una fracción distinta de la que pide el problema.

Lo revelador es cómo se reparten. `Wrong_Fraction` se usa **solo en la pregunta 31777** y `Wrong_fraction` **solo en la pregunta 33471**. Es decir, el equipo que anotó cada pregunta escribió el nombre de forma distinta, y el resultado es que **un mismo error conceptual figura en los datos como dos errores diferentes**.

Sin este hallazgo, las dos etiquetas contarían como ejemplos de errores atados a una única pregunta. Unificadas, pasan a ser un error que sí aparece en dos preguntas distintas, que es justo el tipo de evidencia que la competencia busca.

**Decisión:** la inconsistencia se **documenta pero no se corrige en los datos**. La razón es que la competencia evalúa contra las etiquetas tal como están escritas, de modo que reemplazarlas rompería la correspondencia con el objetivo oficial. La unificación es una decisión que corresponde a la fase de modelado, donde puede aplicarse de forma controlada y revertirse al momento de generar las predicciones.